# Lesson 03 · Natural Transformations

A way to slide one functor onto another, one arrow per object, so coherently that it commutes with everything. This is the idea the whole field was invented to make precise.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-03-natural-transformations.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Naturality by hand

**`natural.py`** — polymorphic functions are natural

In [1]:
def fmap(f, xs):                 # the List functor on arrows
    return [f(x) for x in xs]

def reverse(xs):                 # the component of  reverse : List => List
    return xs[::-1]

inc  = lambda x: x + 1
data = [1, 2, 3, 4]

# Naturality of reverse:   map f . reverse  ==  reverse . map f
assert fmap(inc, reverse(data)) == reverse(fmap(inc, data))
print(fmap(inc, reverse(data)))              # [5, 4, 3, 2]

# safe_head : List => Maybe   (Maybe modelled as a 0-or-1 element list)
def safe_head(xs):
    return [xs[0]] if xs else []

# Naturality of safe_head holds for every list and every f
for xs in ([], [10, 20, 30]):
    assert fmap(inc, safe_head(xs)) == safe_head(fmap(inc, xs))
print("reverse and safe_head are natural")

[5, 4, 3, 2]
reverse and safe_head are natural


**`setup.py`** — carried over from Lessons 1 and 2, plus two small categories

In [2]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

class Functor:
    """F : C -> D, given by a map on objects and a map on arrows."""
    def __init__(self, C, D, on_obj, on_arr):
        self.C, self.D, self.on_obj, self.on_arr = C, D, on_obj, on_arr

    def check(self):
        C, D, Fo, Fa = self.C, self.D, self.on_obj, self.on_arr
        for f in C.ar:                        # typing: F f : F X -> F Y
            X, Y = C.ar[f]
            assert D.ar[Fa[f]] == (Fo[X], Fo[Y])
        for X in C.obj:                       # preserves identities
            assert Fa[C.id[X]] == D.id[Fo[X]]
        for g, f in product(C.ar, repeat=2):  # preserves composition
            if C.composable(g, f):
                assert Fa[C.comp[(g, f)]] == D.comp[(Fa[g], Fa[f])]
        return True

# the category 2 = ( 0 --u--> 1 )
two_ar   = {'i0':(0,0), 'i1':(1,1), 'u':(0,1)}
two_comp = {('i0','i0'):'i0', ('i1','i1'):'i1',
            ('u','i0'):'u',  ('i1','u'):'u'}
Two = FiniteCategory({0,1}, two_ar, two_comp, {0:'i0', 1:'i1'})

# the commuting square:  f: A->B, p: A->C, q: B->D, g: C->D,  d = q.f = g.p
arr = {'iA':('A','A'), 'iB':('B','B'), 'iC':('C','C'), 'iD':('D','D'),
       'f':('A','B'), 'p':('A','C'), 'q':('B','D'), 'g':('C','D'),
       'd':('A','D')}
ident = {'A':'iA', 'B':'iB', 'C':'iC', 'D':'iD'}
comp = {}
for x, (u, v) in arr.items():
    comp[(ident[v], x)] = x          # id . x = x
    comp[(x, ident[u])] = x          # x . id = x
comp[('q','f')] = 'd'                # q . f = d
comp[('g','p')] = 'd'                # g . p = d
Square = FiniteCategory(set('ABCD'), arr, comp, ident)
assert Square.check() and Two.check()

**`nat_transformation.py`** — a naturality checker

In [3]:
class Nat:
    """alpha : F => G, with F, G : C -> D sharing source and target.
       `comp` maps each object X of C to a component arrow alpha_X of D."""
    def __init__(self, F, G, comp):
        self.F, self.G, self.comp = F, G, comp

    def check(self):
        F, G, C, D = self.F, self.G, self.F.C, self.F.D
        for X in C.obj:           # component typing:  alpha_X : F X -> G X
            assert D.ar[self.comp[X]] == (F.on_obj[X], G.on_obj[X])
        for f in C.ar:            # naturality:  G f . alpha_X == alpha_Y . F f
            X, Y = C.ar[f]
            assert D.comp[(G.on_arr[f], self.comp[X])] == \
                   D.comp[(self.comp[Y], F.on_arr[f])]
        return True

# F and G send the category 2 to two opposite edges of the square; a natural
# transformation between them is EXACTLY a commuting square (components p, q).
F = Functor(Two, Square, {0:'A', 1:'B'}, {'i0':'iA', 'i1':'iB', 'u':'f'})
G = Functor(Two, Square, {0:'C', 1:'D'}, {'i0':'iC', 'i1':'iD', 'u':'g'})
alpha = Nat(F, G, comp={0:'p', 1:'q'})
print(alpha.check())

True


**`more_naturality.py`** — the examples of §4 and §7, by brute force

In [4]:
from itertools import combinations, product

def subsets(X):
    X = list(X)
    return [frozenset(c) for r in range(len(X) + 1)
                         for c in combinations(X, r)]

P     = lambda f: (lambda S: frozenset(f(x) for x in S))   # direct image
eta   = lambda x: frozenset([x])                           # singleton
union = lambda SS: frozenset(x for S in SS for x in S)

X = {1, 2, 3}
f = {1: 'a', 2: 'b', 3: 'a'}.get
families = subsets(subsets(X))            # all 256 sets of subsets of X
assert all(P(f)(eta(x)) == eta(f(x)) for x in X)
assert all(P(f)(union(SS)) == union(P(P(f))(SS)) for SS in families)
print("singleton and union are natural;", len(families), "families checked")

def equivariant(s, t):
    "all functions a with  a . s == t . a,  for involutions s and t"
    S, T = sorted(s), sorted(t)
    maps = [dict(zip(S, vals)) for vals in product(T, repeat=len(S))]
    return [a for a in maps if all(a[s[x]] == t[a[x]] for x in S)]

s  = {1: 2, 2: 1, 3: 3}                   # on S   = {1, 2, 3}
s1 = {'a': 'b', 'b': 'a'}                 # on S'  = {a, b}
s2 = {'a': 'b', 'b': 'a', 'c': 'c'}       # on S'' = {a, b, c}
print("S -> S' :", len(equivariant(s, s1)), "equivariant maps")
print("S -> S'':", len(equivariant(s, s2)), "equivariant maps")

singleton and union are natural; 256 families checked
S -> S' : 0 equivariant maps
S -> S'': 3 equivariant maps


## Exercise 3

**`sorted_not_natural.py`**

In [5]:
dup = lambda xs: [x for x in xs for _ in (0, 1)]
neg = lambda x: -x
assert fmap(neg, dup([1, 2])) == dup(fmap(neg, [1, 2]))   # natural

lhs = fmap(neg, sorted([1, 2]))     # [-1, -2]
rhs = sorted(fmap(neg, [1, 2]))     # [-2, -1]
print(lhs, rhs, lhs == rhs)         # [-1, -2] [-2, -1] False

[-1, -2] [-2, -1] False


## Exercise 4

**`dedupe_not_natural.py`**

In [6]:
take2  = lambda xs: xs[:2]
dedupe = lambda xs: list(dict.fromkeys(xs))
const0 = lambda x: 0

for f in (str, lambda x: -x):
    for xs in ([], [1], [3, 1, 2], [5, 5, 1]):
        assert fmap(f, take2(xs)) == take2(fmap(f, xs))   # natural

lhs = fmap(const0, dedupe([1, 2]))    # dedupe sees 1 != 2 ...
rhs = dedupe(fmap(const0, [1, 2]))    # ... but const0 merges them
print(lhs, rhs, lhs == rhs)

[0, 0] [0] False


## Exercise 13

**`broken_component.py`**

In [7]:
bad = Nat(F, G, comp={0:'p', 1:'f'})
try:
    bad.check()
except AssertionError:
    print("rejected: component at 1 is mistyped")

rejected: component at 1 is mistyped
